In [ ]:
import random
from tqdm import tqdm
from collections import defaultdict
def get_available_nodes(graph, source, hop=1):
    if hop == 0:
        return set(graph.keys()) - {source}
    current = {source}
    visited = {source}
    available_nodes = set()
    for _ in range(hop):
        next_level = set()
        for node in current:
            neighbors = graph[node]
            available_nodes.update(neighbors)
            for neighbor in neighbors:
                if neighbor not in visited:
                    visited.add(neighbor)
                    next_level.add(neighbor)
        current = next_level
        if not current:
            break
    available_nodes.discard(source)
    return available_nodes

def simulate(graph, victim, originator, available_nodes, p=1):
    if originator not in available_nodes:
        raise ValueError("Originator not in victim neighborhood")
    victim_neighbors = graph[victim]
    remaining = set(victim_neighbors)  # not yet infected
    current = {originator}
    visited = {originator}
    time = 0
    spread_time = 0
    if originator in remaining:
        remaining.remove(originator)

    while current and remaining:
        next_level = set()
        for node in current:
            neighbors = available_nodes & graph[node]
            for neighbor in neighbors:
                if neighbor not in visited and random.random() <= p:
                    visited.add(neighbor)
                    next_level.add(neighbor)
                    if neighbor in remaining:
                        remaining.remove(neighbor)
                        spread_time = time + 1  # last infection time updates
        current = next_level
        time += 1
    victim_degree = len(victim_neighbors)
    infected = victim_neighbors - remaining
    spread_factor = len(infected) / victim_degree if victim_degree > 0 else 0

    return spread_time, spread_factor

def run_simulation_by_k(graph, hop=1, p=1):
    data_by_k = defaultdict(lambda: defaultdict(list))
    for victim, victim_node in graph.items():
        available_nodes = get_available_nodes(graph, victim, hop=hop)
        for originator in victim_node:
            data = simulate(graph, victim, originator, available_nodes, p)
            spread_time, spread_factor = data[0], data[1]
            k = len(victim_node)
            data_by_k[k]["spread_time"].append(spread_time)
            data_by_k[k]["spread_factor"].append(spread_factor)
    return data_by_k

In [ ]:
import pickle
import os
def load_graphs(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def to_dict(obj):
    if isinstance(obj, defaultdict):
        return {k: to_dict(v) for k, v in obj.items()}
    return obj

In [ ]:
m = 5
for size in [20, 25, 30, 35, 40, 45, 50]:
    if not os.path.exists(f"fig_data/fig03/fig03_m05_N{size:02d}K.pkl"):
        result = defaultdict(lambda: defaultdict(list))
        BA_path = f"data/ba/ba_network_m{m:02d}_N{size:03d}K.pkl"
        BA_graphs = load_graphs(BA_path)
        print(f"Running simulation for size={size}K:")
        for graph in tqdm(BA_graphs):
            graph_result = run_simulation_by_k(graph)
            for k, metrics in graph_result.items():
                for metric, values in metrics.items():
                    result[k][metric].extend(values)
        result = to_dict(result)
        del BA_graphs
        with open(f"fig_data/fig03/fig03_m05_N{size:02d}K.pkl", "wb") as f:
            pickle.dump(result, f)
        del result

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
k0 = []
for size in [20, 25, 30, 35, 40, 45, 50]:
    BA_data_by_k = load_graphs(f"fig_data/fig03/fig03_m05_N{size:02d}K.pkl")
    BA_plot_data_by_k = defaultdict(dict) 
    for k in BA_data_by_k: 
        BA_plot_data_by_k[k]["spread_factor"] = sum(BA_data_by_k[k]["spread_factor"]) / len(BA_data_by_k[k]["spread_factor"])
    ks = sorted(BA_plot_data_by_k)
    spread_factors = [BA_plot_data_by_k[k]["spread_factor"]for k in ks]
    idx = spread_factors.index(min(spread_factors))
    k0.append(ks[idx])

k0 = np.array(k0)
N = np.array([20, 25, 30, 35, 40, 45, 50])
x = np.log(np.log(N))
y = np.log(k0)
a, b = np.polyfit(x, y, 1)
# smooth fitted line
x_line = np.linspace(x.min(), x.max(), 200)
y_line = a * x_line + b

# plot
print("Fig 3a")
print(f"A: {a}, B: {b}")
plt.figure(figsize=(8, 6))
plt.scatter(x, y, label="Data")
plt.plot(x_line, y_line, "k--", label=fr"Fit: $y={a:.3f}x+{b:.3f}$")
plt.xlabel(r"$\ln(\ln(N))$")
plt.ylabel(r"$\ln(k_0)$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
size = 10
for m in [3, 5, 7, 9, 11, 13, 15]:
    if not os.path.exists(f"fig_data/fig03/fig03_m{m:02d}_N{size:02d}K.pkl"):
        result = defaultdict(lambda: defaultdict(list))
        BA_path = f"data/ba/ba_network_m{m:02d}_N{size:03d}K.pkl"
        BA_graphs = load_graphs(BA_path)
        print(f"Running simulation for m={m}:")
        for graph in tqdm(BA_graphs):
            graph_result = run_simulation_by_k(graph)
            for k, metrics in graph_result.items():
                for metric, values in metrics.items():
                    result[k][metric].extend(values)
        result = to_dict(result)
        del BA_graphs
        with open(f"fig_data/fig03/fig03_m{m:02d}_N{size:02d}K.pkl", "wb") as f:
            pickle.dump(result, f)
        del result

In [ ]:
k0 = []
for m in [3, 5, 7, 9, 11, 13, 15]:
    BA_data_by_k = load_graphs(f"fig_data/fig03/fig03_m{m:02d}_N10K.pkl")
    BA_plot_data_by_k = defaultdict(dict) 
    for k in BA_data_by_k: 
        BA_plot_data_by_k[k]["spread_factor"] = sum(BA_data_by_k[k]["spread_factor"]) / len(BA_data_by_k[k]["spread_factor"])
    ks = sorted(BA_plot_data_by_k)
    spread_factors = [BA_plot_data_by_k[k]["spread_factor"]for k in ks]
    idx = spread_factors.index(min(spread_factors))
    k0.append(ks[idx])

k0 = np.array(k0)
M = np.array([3, 5, 7, 9, 11, 13, 15])
x = np.log(np.log(M))
y = np.log(k0)
a, b = np.polyfit(x, y, 1)
# smooth fitted line
x_line = np.linspace(x.min(), x.max(), 200)
y_line = a * x_line + b

# plot
print("Fig 3b")
print(f"A: {a}, B: {b}")
plt.figure(figsize=(8, 6))
plt.scatter(x, y, label="Data")
plt.plot(x_line, y_line, "k--", label=fr"Fit: $y={a:.3f}x+{b:.3f}$")
plt.xlabel(r"$\ln(\ln(m))$")
plt.ylabel(r"$\ln(k_0)$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
